# 🙂 Тема 3. Находим лица на видео

**Часть 1** — прямоугольник: нарисовать рамку и вырезать кусок картинки.
**Часть 2** — детектор лиц OpenCV на одной фотографии.
**Часть 3** — несколько лиц сразу.
**Часть 4** — лица на видео: рамки на каждом кадре и видео «только из лиц».
**Часть 5** — КТ: то же самое на своём видео.

**Как устроено каждое задание**
1. Короткое объяснение и пример.
2. **Задание ✏️** — замените `None` на свой код и запустите (**Shift + Enter**).
3. **🧪 Проверка** — запустите ячейку под заданием: ✅ или ❌ и подсказка, что не так.
4. Застряли — откройте **💡 Подсказку**, потом **🆘 Почти готовый ответ**.

> Сначала: **Файл → Сохранить копию на Диске**, иначе работа пропадёт. Шпаргалка — в конце.

In [ ]:
# Подключаем библиотеки и скачиваем фотографию (качает сервер Colab, а не ваш интернет)
import os
import cv2
import numpy as np
from google.colab.patches import cv2_imshow

!wget -q -nc https://raw.githubusercontent.com/opencv/opencv/4.x/samples/data/lena.jpg
img = cv2.imread("lena.jpg")
print("Готово! Размер фото:", img.shape)
cv2_imshow(img)

In [ ]:
#@title ▶️ Запустите: помощники для видео { display-mode: "form" }
# show_video(кадры)          — показать список кадров плеером (в Colab нет окна для видео)
# make_face_video(faces=1)   — учебное видео: по кадру ездит 1 или 2 лица
# record_webcam(5)           — записать 5 секунд с веб-камеры (браузер спросит разрешение)
# read_video(путь)           — прочитать видео в список кадров
import base64, subprocess, shutil
from IPython.display import HTML, Javascript, display

def show_video(frames, fps=10, width=480):
    if len(frames) == 0 or any(f is None for f in frames):
        print("⚠️ Показывать нечего: список пуст или в нём None")
        return
    prep = []
    for f in frames:
        f = np.clip(np.asarray(f), 0, 255).astype(np.uint8)
        if f.ndim == 2:
            f = cv2.cvtColor(f, cv2.COLOR_GRAY2BGR)
        h, w = f.shape[:2]
        prep.append(np.ascontiguousarray(f[: h - h % 2, : w - w % 2]))
    h, w = prep[0].shape[:2]
    if any(p.shape != prep[0].shape for p in prep):
        print("⚠️ Кадры разного размера. Приведите их к одному размеру через cv2.resize")
        return
    cmd = ["ffmpeg", "-y", "-loglevel", "error", "-f", "rawvideo", "-pix_fmt", "bgr24", "-s", f"{w}x{h}",
           "-r", str(fps), "-i", "-", "-c:v", "libx264", "-pix_fmt", "yuv420p", "-crf", "30",
           "-movflags", "+faststart", "-f", "mp4", "-"]
    data = subprocess.run(cmd, input=b"".join(f.tobytes() for f in prep), capture_output=True).stdout
    b64 = base64.b64encode(data).decode()
    display(HTML(f'<video width="{width}" controls autoplay loop muted><source src="data:video/mp4;base64,{b64}" type="video/mp4"></video>'))

def make_face_video(faces=1, n_frames=40):
    lena = cv2.imread("lena.jpg")
    face = cv2.resize(lena[170:410, 190:420], (130, 130))       # кусок фото с лицом
    grad = np.linspace(120, 200, 480).astype(np.uint8)
    background = np.dstack([np.tile(grad, (320, 1))] * 3)       # серый фон 480×320
    frames = []
    for i in range(n_frames):
        f = background.copy()
        x = 10 + i * 8
        f[15:145, x:x + 130] = face                              # первое лицо едет вправо
        if faces == 2:
            x2 = 340 - i * 8
            f[175:305, x2:x2 + 130] = cv2.flip(face, 1)          # второе — влево
        frames.append(f)
    return frames

def record_webcam(seconds=5, filename="webcam.webm"):
    from google.colab.output import eval_js
    display(Javascript("""
    async function recordVideo(ms) {
      const stream = await navigator.mediaDevices.getUserMedia({video: true, audio: false});
      const video = document.createElement('video');
      video.srcObject = stream; video.muted = true; video.width = 320;
      document.body.appendChild(video);
      await video.play();
      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
      const recorder = new MediaRecorder(stream, {mimeType: 'video/webm'});
      const chunks = [];
      recorder.ondataavailable = e => chunks.push(e.data);
      const stopped = new Promise(resolve => recorder.onstop = resolve);
      recorder.start();
      await new Promise(resolve => setTimeout(resolve, ms));
      recorder.stop(); await stopped;
      stream.getTracks().forEach(t => t.stop()); video.remove();
      const bytes = new Uint8Array(await new Blob(chunks).arrayBuffer());
      let binary = '';
      for (let i = 0; i < bytes.length; i += 0x8000) binary += String.fromCharCode.apply(null, bytes.subarray(i, i + 0x8000));
      return btoa(binary);
    }"""))
    print(f"Запись {seconds} с… Смотрите в камеру!")
    data = eval_js(f"recordVideo({int(seconds * 1000)})")
    open(filename, "wb").write(base64.b64decode(data))
    fixed = "webcam.mp4"                                         # у записи браузера нет нормального FPS — чиним
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", filename, "-r", "10", "-an", fixed], check=True)
    return fixed

def read_video(path, max_frames=60, width=480):
    cap = cv2.VideoCapture(path)
    frames = []
    while len(frames) < max_frames:
        ok, frame = cap.read()
        if not ok:
            break
        h, w = frame.shape[:2]
        frames.append(cv2.resize(frame, (width, int(h * width / w))))
    cap.release()
    print("Прочитано кадров:", len(frames))
    return frames

print("✅ Помощники готовы: show_video, make_face_video, record_webcam, read_video")

---
# Часть 1. Прямоугольник

Любой детектор отвечает не «да/нет», а **где**: прямоугольник `(x, y, w, h)`.
* `x, y` — левый верхний угол, `w` — ширина, `h` — высота.
* Правый нижний угол — `(x + w, y + h)`.

Рамка: `cv2.rectangle(картинка, (x, y), (x + w, y + h), цвет, толщина)`. Цвет в BGR: зелёный — `(0, 255, 0)`.

In [ ]:
demo = img.copy()                                  # копия, чтобы не испортить оригинал
cv2.rectangle(demo, (50, 50), (50 + 100, 50 + 60), (0, 255, 0), 3)
cv2_imshow(demo)

### ✏️ Задание 1. Нарисуйте на копии `box_img` зелёную рамку толщиной 3: `x = 200`, `y = 200`, `w = 180`, `h = 180`

**Что должно получиться:** рамка вокруг лица.

In [ ]:
box_img = img.copy()
x, y, w, h = 200, 200, 180, 180
# ваш код: cv2.rectangle(...)

In [ ]:
#@title 🧪 Проверка задания 1 { display-mode: "form" }
def _check(tests):
    for i, (name, cond, hint) in enumerate(tests, 1):
        try:
            passed = bool(cond())
        except Exception:
            passed = False
        if not passed:
            print(f"❌ Тест {i} из {len(tests)}: {name}")
            print(f"   💡 {hint}")
            return False
        print(f"✅ Тест {i}: {name}")
    print(f"\n🎉 Задание 1 выполнено! Пройдено тестов: {len(tests)} из {len(tests)}")
    return True

if _check([
    ('на копии что-то нарисовано', lambda: not np.array_equal(box_img, img), 'Рисуйте на box_img, а не на img'),
    ('левый верхний угол зелёный', lambda: tuple(box_img[y, x]) == (0, 255, 0), 'Первая точка — (x, y), цвет (0, 255, 0)'),
    ('правый нижний угол на месте', lambda: tuple(box_img[y + h, x + w]) == (0, 255, 0), 'Вторая точка — (x + w, y + h)'),
    ('оригинал не тронут', lambda: tuple(img[y, x]) != (0, 255, 0), 'Рисуйте на копии: box_img = img.copy()'),
]):
    cv2_imshow(box_img)

<details><summary>💡 Подсказка</summary>

Две точки: левый верхний угол `(x, y)` и правый нижний `(x + w, y + h)`.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
cv2.rectangle(box_img, (x, y), (x + ___, y + ___), (0, 255, 0), 3)
```
</details>

**Вырезать** кусок — срез массива. Сначала строки (`y`), потом столбцы (`x`): `кусок = img[y:y + h, x:x + w]`.

> ⚠️ В срезе **сначала y**, а в `cv2.rectangle` точки пишутся **(x, y)**. Самая частая ошибка темы.

### ✏️ Задание 2. Вырежьте из `img` прямоугольник из задания 1 в переменную `piece`

In [ ]:
piece = None

In [ ]:
#@title 🧪 Проверка задания 2 { display-mode: "form" }
def _check(tests):
    for i, (name, cond, hint) in enumerate(tests, 1):
        try:
            passed = bool(cond())
        except Exception:
            passed = False
        if not passed:
            print(f"❌ Тест {i} из {len(tests)}: {name}")
            print(f"   💡 {hint}")
            return False
        print(f"✅ Тест {i}: {name}")
    print(f"\n🎉 Задание 2 выполнено! Пройдено тестов: {len(tests)} из {len(tests)}")
    return True

if _check([
    ('кусок вырезан', lambda: piece is not None, 'Замените None на срез img[...]'),
    ('размер 180×180', lambda: piece.shape == (180, 180, 3), 'Срез: img[y:y + h, x:x + w] — сначала y'),
    ('вырезано нужное место', lambda: np.array_equal(piece, img[200:380, 200:380]), 'Проверьте порядок: сначала y, потом x'),
]):
    cv2_imshow(piece)

<details><summary>💡 Подсказка</summary>

Сначала строки от `y` до `y + h`, потом столбцы от `x` до `x + w`.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
piece = img[y:y + ___, x:x + ___]
```
</details>

---
# Часть 2. Детектор лиц

В OpenCV уже есть обученный детектор лиц — **каскад Хаара** (2001 год). Это файл `.xml` с тысячами простых проверок вида «область глаз темнее щёк», «переносица светлее глаз».

Детектор двигает окошко по картинке, на разных масштабах, и для каждого окошка прогоняет проверки ступенями. Не прошло первую ступень — дальше не смотрим. Поэтому он быстрый.

Файлы каскадов лежат в папке `cv2.data.haarcascades`.

In [ ]:
print(cv2.data.haarcascades)
print([f for f in os.listdir(cv2.data.haarcascades) if "face" in f or "eye" in f])

### ✏️ Задание 3. Загрузите детектор лиц `haarcascade_frontalface_default.xml` в переменную `face_cascade`

In [ ]:
path = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
face_cascade = None   # cv2.CascadeClassifier(путь)

In [ ]:
#@title 🧪 Проверка задания 3 { display-mode: "form" }
def _check(tests):
    for i, (name, cond, hint) in enumerate(tests, 1):
        try:
            passed = bool(cond())
        except Exception:
            passed = False
        if not passed:
            print(f"❌ Тест {i} из {len(tests)}: {name}")
            print(f"   💡 {hint}")
            return False
        print(f"✅ Тест {i}: {name}")
    print(f"\n🎉 Задание 3 выполнено! Пройдено тестов: {len(tests)} из {len(tests)}")
    return True

if _check([
    ('детектор создан', lambda: face_cascade is not None, 'Замените None на cv2.CascadeClassifier(path)'),
    ('файл прочитан (детектор не пустой)', lambda: not face_cascade.empty(), 'Проверьте имя файла — без опечаток'),
]):
    print("Детектор готов 👍")

<details><summary>💡 Подсказка</summary>

`cv2.CascadeClassifier(path)` — путь уже записан в переменной `path`.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
face_cascade = cv2.CascadeClassifier(___)
```
</details>

Детектор работает с **серой** картинкой: ему важна яркость, а не цвет.

### ✏️ Задание 4. Переведите `img` в серый цвет: переменная `gray`

In [ ]:
gray = None

In [ ]:
#@title 🧪 Проверка задания 4 { display-mode: "form" }
def _check(tests):
    for i, (name, cond, hint) in enumerate(tests, 1):
        try:
            passed = bool(cond())
        except Exception:
            passed = False
        if not passed:
            print(f"❌ Тест {i} из {len(tests)}: {name}")
            print(f"   💡 {hint}")
            return False
        print(f"✅ Тест {i}: {name}")
    print(f"\n🎉 Задание 4 выполнено! Пройдено тестов: {len(tests)} из {len(tests)}")
    return True

if _check([
    ('картинка серая (2 измерения)', lambda: gray.ndim == 2, 'cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)'),
    ('размер как у фото', lambda: gray.shape == img.shape[:2], 'Переводите именно img'),
]):
    cv2_imshow(gray)

<details><summary>💡 Подсказка</summary>

Как в теме 1: `cv2.cvtColor(картинка, cv2.COLOR_BGR2GRAY)`.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
gray = cv2.cvtColor(img, cv2.___)
```
</details>

Главная функция: `faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5)`.

Она возвращает **список прямоугольников** `(x, y, w, h)` — по одному на каждое найденное лицо.
* `scaleFactor=1.1` — на сколько уменьшать картинку на каждом шаге (ищем лица разного размера).
* `minNeighbors=5` — сколько «соседних» срабатываний нужно, чтобы поверить, что это лицо. Меньше — больше ложных рамок.

### ✏️ Задание 5. Найдите лица на `gray` в переменную `faces`

In [ ]:
faces = None

In [ ]:
#@title 🧪 Проверка задания 5 { display-mode: "form" }
def _check(tests):
    for i, (name, cond, hint) in enumerate(tests, 1):
        try:
            passed = bool(cond())
        except Exception:
            passed = False
        if not passed:
            print(f"❌ Тест {i} из {len(tests)}: {name}")
            print(f"   💡 {hint}")
            return False
        print(f"✅ Тест {i}: {name}")
    print(f"\n🎉 Задание 5 выполнено! Пройдено тестов: {len(tests)} из {len(tests)}")
    return True

if _check([
    ('что-то найдено', lambda: faces is not None and len(faces) > 0, 'face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5)'),
    ('найдено ровно одно лицо', lambda: len(faces) == 1, 'Передайте gray, scaleFactor=1.1, minNeighbors=5'),
]):
    print("Найдено лиц:", len(faces))
    print("Прямоугольник (x, y, w, h):", faces[0])

<details><summary>💡 Подсказка</summary>

Метод вызывается у детектора: `face_cascade.detectMultiScale(...)`.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
faces = face_cascade.detectMultiScale(gray, scaleFactor=___, minNeighbors=___)
```
</details>

Лиц может быть несколько, поэтому рамки рисуем **циклом**:
```python
for (x, y, w, h) in faces:
    cv2.rectangle(...)
```

### ✏️ Задание 6. Нарисуйте зелёные рамки (толщина 3) вокруг всех лиц на копии `found`

In [ ]:
found = img.copy()
for (x, y, w, h) in faces:
    pass   # ← здесь cv2.rectangle(...)

In [ ]:
#@title 🧪 Проверка задания 6 { display-mode: "form" }
def _check(tests):
    for i, (name, cond, hint) in enumerate(tests, 1):
        try:
            passed = bool(cond())
        except Exception:
            passed = False
        if not passed:
            print(f"❌ Тест {i} из {len(tests)}: {name}")
            print(f"   💡 {hint}")
            return False
        print(f"✅ Тест {i}: {name}")
    print(f"\n🎉 Задание 6 выполнено! Пройдено тестов: {len(tests)} из {len(tests)}")
    return True

if _check([
    ('рамка нарисована', lambda: not np.array_equal(found, img), 'Внутри цикла вызовите cv2.rectangle для found'),
    ('рамка там, где лицо', lambda: tuple(found[faces[0][1], faces[0][0]]) == (0, 255, 0), 'Точки (x, y) и (x + w, y + h), цвет (0, 255, 0)'),
]):
    cv2_imshow(found)

<details><summary>💡 Подсказка</summary>

Внутри цикла — то же, что в задании 1.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
for (x, y, w, h) in faces:
    cv2.rectangle(found, (x, y), (x + w, y + h), (0, 255, 0), ___)
```
</details>

### ✏️ Задание 7. Вырежьте первое найденное лицо в `face` и уменьшите его до 200×200 в `face200`

**Что должно получиться:** крупное лицо 200×200. Зачем один размер — увидим в части 4.

In [ ]:
x, y, w, h = faces[0]
face = None
face200 = None

In [ ]:
#@title 🧪 Проверка задания 7 { display-mode: "form" }
def _check(tests):
    for i, (name, cond, hint) in enumerate(tests, 1):
        try:
            passed = bool(cond())
        except Exception:
            passed = False
        if not passed:
            print(f"❌ Тест {i} из {len(tests)}: {name}")
            print(f"   💡 {hint}")
            return False
        print(f"✅ Тест {i}: {name}")
    print(f"\n🎉 Задание 7 выполнено! Пройдено тестов: {len(tests)} из {len(tests)}")
    return True

if _check([
    ('лицо вырезано', lambda: face is not None and face.shape[:2] == (faces[0][3], faces[0][2]), 'Срез как в задании 2: img[y:y + h, x:x + w]'),
    ('размер 200×200', lambda: face200.shape == (200, 200, 3), 'cv2.resize(face, (200, 200))'),
]):
    cv2_imshow(face200)

<details><summary>💡 Подсказка</summary>

Сначала срез, потом `cv2.resize(кусок, (ширина, высота))`.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
face = img[y:y + h, x:x + ___]
face200 = cv2.resize(face, (___, ___))
```
</details>

---
# Часть 3. Несколько лиц

Соберём «групповое фото»: три копии склеим в ряд функцией `np.hstack`. Одну отразим, другую затемним.

In [ ]:
small = cv2.resize(img, (300, 300))
group = np.hstack([small, cv2.flip(small, 1), cv2.convertScaleAbs(small, alpha=0.7)])
cv2_imshow(group)

### ✏️ Задание 8. Найдите все лица на `group`: список в `group_faces`, их количество — в `n_faces`

In [ ]:
group_gray = None
group_faces = None
n_faces = None

In [ ]:
#@title 🧪 Проверка задания 8 { display-mode: "form" }
def _check(tests):
    for i, (name, cond, hint) in enumerate(tests, 1):
        try:
            passed = bool(cond())
        except Exception:
            passed = False
        if not passed:
            print(f"❌ Тест {i} из {len(tests)}: {name}")
            print(f"   💡 {hint}")
            return False
        print(f"✅ Тест {i}: {name}")
    print(f"\n🎉 Задание 8 выполнено! Пройдено тестов: {len(tests)} из {len(tests)}")
    return True

if _check([
    ('серая картинка', lambda: group_gray.ndim == 2, 'Сначала cvtColor(group, cv2.COLOR_BGR2GRAY)'),
    ('лица найдены', lambda: len(group_faces) > 0, 'detectMultiScale(group_gray, scaleFactor=1.1, minNeighbors=5)'),
    ('найдено 3 лица', lambda: n_faces == 3, 'n_faces = len(group_faces); проверьте minNeighbors=5'),
]):
    print("Лиц на фото:", n_faces)

<details><summary>💡 Подсказка</summary>

Те же два шага, что в заданиях 4 и 5, только для `group`. Количество — `len(...)`.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
group_gray = cv2.cvtColor(group, cv2.COLOR_BGR2GRAY)
group_faces = face_cascade.detectMultiScale(___, scaleFactor=1.1, minNeighbors=5)
n_faces = len(___)
```
</details>

### ✏️ Задание 9. Нарисуйте рамки вокруг всех лиц на `group_boxes` и напишите сверху `Faces: 3` (число берите из `n_faces`)

**Что должно получиться:** это и есть вторая КТ — «несколько лиц сразу», только на фото.

In [ ]:
group_boxes = group.copy()
# 1) цикл по group_faces с cv2.rectangle
# 2) cv2.putText(group_boxes, f"Faces: {n_faces}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

In [ ]:
#@title 🧪 Проверка задания 9 { display-mode: "form" }
def _check(tests):
    for i, (name, cond, hint) in enumerate(tests, 1):
        try:
            passed = bool(cond())
        except Exception:
            passed = False
        if not passed:
            print(f"❌ Тест {i} из {len(tests)}: {name}")
            print(f"   💡 {hint}")
            return False
        print(f"✅ Тест {i}: {name}")
    print(f"\n🎉 Задание 9 выполнено! Пройдено тестов: {len(tests)} из {len(tests)}")
    return True

if _check([
    ('на картинке что-то нарисовано', lambda: not np.array_equal(group_boxes, group), 'Рисуйте на group_boxes'),
    ('рамки у всех лиц', lambda: all(tuple(group_boxes[y, x]) == (0, 255, 0) for (x, y, w, h) in group_faces), 'Цикл по group_faces, в нём cv2.rectangle'),
    ('есть надпись сверху слева', lambda: (group_boxes[5:35, 10:150] != group[5:35, 10:150]).any(), 'cv2.putText(..., (10, 30), ...)'),
]):
    cv2_imshow(group_boxes)

<details><summary>💡 Подсказка</summary>

Цикл — как в задании 6. Текст — `cv2.putText(картинка, текст, (x, y), шрифт, размер, цвет, толщина)`.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
for (x, y, w, h) in group_faces:
    cv2.rectangle(group_boxes, (x, y), (x + w, y + h), (0, 255, 0), 3)
cv2.putText(group_boxes, f"Faces: {n_faces}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
```
</details>

### 🔬 Эксперимент: что делает `minNeighbors`
Запустите и посмотрите, сколько рамок при разных значениях.

In [ ]:
for mn in [0, 1, 3, 5, 10]:
    found_n = len(face_cascade.detectMultiScale(group_gray, scaleFactor=1.1, minNeighbors=mn))
    print(f"minNeighbors = {mn:2d} → рамок: {found_n}")

**Мой ответ** (дважды щёлкните, чтобы написать): почему при `minNeighbors = 0` рамок так много, а при `1` появилось лишнее «лицо»?

…

---
# Часть 4. Лица на видео

Видео — это много кадров. Значит, делаем то же самое **в цикле по кадрам**. Для начала учебное видео, где по кадру ездит лицо.

In [ ]:
video = make_face_video(faces=1)
print("Кадров:", len(video), "размер кадра:", video[0].shape)
show_video(video)

### ✏️ Задание 10. Нарисуйте рамку вокруг лица на каждом кадре: список `boxed`

**Что должно получиться:** рамка едет вместе с лицом.

In [ ]:
boxed = []
for frame in video:
    frame = frame.copy()
    g = None          # серый кадр
    found_here = []   # лица на кадре
    for (x, y, w, h) in found_here:
        cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 3)
    boxed.append(frame)

In [ ]:
#@title 🧪 Проверка задания 10 { display-mode: "form" }
def _check(tests):
    for i, (name, cond, hint) in enumerate(tests, 1):
        try:
            passed = bool(cond())
        except Exception:
            passed = False
        if not passed:
            print(f"❌ Тест {i} из {len(tests)}: {name}")
            print(f"   💡 {hint}")
            return False
        print(f"✅ Тест {i}: {name}")
    print(f"\n🎉 Задание 10 выполнено! Пройдено тестов: {len(tests)} из {len(tests)}")
    return True

if _check([
    ('кадров столько же', lambda: len(boxed) == len(video), 'Добавляйте каждый кадр в boxed'),
    ('на кадрах есть зелёные рамки', lambda: sum((f == (0, 255, 0)).all(axis=2).sum() > 100 for f in boxed) == len(video), 'Внутри цикла: cvtColor → detectMultiScale → rectangle'),
]):
    show_video(boxed)

<details><summary>💡 Подсказка</summary>

Внутри цикла — задания 4 и 5, только вместо `img` кадр `frame`.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
g = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
found_here = face_cascade.detectMultiScale(g, scaleFactor=1.1, minNeighbors=5)
```
</details>

Теперь видео **только из лиц**: с каждого кадра вырезаем лицо.

> ⚠️ Лица на разных кадрах разного размера, а **в видео все кадры должны быть одного размера** — иначе файл не запишется. Поэтому каждое лицо приводим к 200×200, как в задании 7.

### ✏️ Задание 11. Соберите список `face_frames`: с каждого кадра вырезанное лицо 200×200

**Что должно получиться:** лицо стоит на месте, а фон за ним «едет».

In [ ]:
face_frames = []
for frame in video:
    g = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    found_here = face_cascade.detectMultiScale(g, scaleFactor=1.1, minNeighbors=5)
    for (x, y, w, h) in found_here:
        pass   # вырежьте лицо, уменьшите до 200×200 и добавьте в face_frames

In [ ]:
#@title 🧪 Проверка задания 11 { display-mode: "form" }
def _check(tests):
    for i, (name, cond, hint) in enumerate(tests, 1):
        try:
            passed = bool(cond())
        except Exception:
            passed = False
        if not passed:
            print(f"❌ Тест {i} из {len(tests)}: {name}")
            print(f"   💡 {hint}")
            return False
        print(f"✅ Тест {i}: {name}")
    print(f"\n🎉 Задание 11 выполнено! Пройдено тестов: {len(tests)} из {len(tests)}")
    return True

if _check([
    ('лица собраны', lambda: len(face_frames) > 0, 'В цикле: face = frame[y:y + h, x:x + w], потом append'),
    ('лицо с каждого кадра', lambda: len(face_frames) == len(video), 'На каждом кадре одно лицо — должно получиться столько же'),
    ('все 200×200', lambda: all(f.shape == (200, 200, 3) for f in face_frames), 'cv2.resize(face, (200, 200)) перед append'),
]):
    show_video(face_frames, width=200)

<details><summary>💡 Подсказка</summary>

Задание 7 внутри цикла: срез → `cv2.resize` → `face_frames.append(...)`.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
face = frame[y:y + h, x:x + w]
face_frames.append(cv2.resize(face, (___, ___)))
```
</details>

Сохраняем в файл. Папку создаёт `os.makedirs("video", exist_ok=True)`. Запись видео — как в теме 1:
```python
out = cv2.VideoWriter("video/faces.mp4", cv2.VideoWriter_fourcc(*"mp4v"), fps, (ширина, высота))
out.write(кадр)   # для каждого кадра
out.release()     # обязательно, иначе файл битый
```

### ✏️ Задание 12. Сохраните `face_frames` в `video/faces.mp4` (FPS = 10, размер `(200, 200)`)

**Что должно получиться:** это первая КТ — «записать только лица», пока на учебном видео.

In [ ]:
os.makedirs("video", exist_ok=True)
out = None
# цикл: out.write(f) для каждого f из face_frames, потом out.release()

In [ ]:
#@title 🧪 Проверка задания 12 { display-mode: "form" }
def _check(tests):
    for i, (name, cond, hint) in enumerate(tests, 1):
        try:
            passed = bool(cond())
        except Exception:
            passed = False
        if not passed:
            print(f"❌ Тест {i} из {len(tests)}: {name}")
            print(f"   💡 {hint}")
            return False
        print(f"✅ Тест {i}: {name}")
    print(f"\n🎉 Задание 12 выполнено! Пройдено тестов: {len(tests)} из {len(tests)}")
    return True

if _check([
    ('файл создан', lambda: os.path.exists('video/faces.mp4'), 'cv2.VideoWriter("video/faces.mp4", ...)'),
    ('файл не пустой', lambda: os.path.getsize('video/faces.mp4') > 1000, 'Запишите кадры циклом и вызовите out.release()'),
    ('в файле все кадры', lambda: int(cv2.VideoCapture('video/faces.mp4').get(cv2.CAP_PROP_FRAME_COUNT)) == len(face_frames), 'Размер в VideoWriter — (200, 200), как у кадров'),
]):
    print("Файл video/faces.mp4:", os.path.getsize("video/faces.mp4") // 1024, "КБ")

<details><summary>💡 Подсказка</summary>

Размер в `VideoWriter` — `(ширина, высота)` и должен совпадать с кадрами.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
out = cv2.VideoWriter("video/faces.mp4", cv2.VideoWriter_fourcc(*"mp4v"), 10, (200, 200))
for f in face_frames:
    out.___(f)
out.___()
```
</details>

### Два лица на видео
`make_face_video(faces=2)` — по кадру едут два лица навстречу друг другу.

### ✏️ Задание 13. Посчитайте, сколько лиц найдено на каждом кадре: список `counts`

In [ ]:
video2 = make_face_video(faces=2)
counts = []
for frame in video2:
    g = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    found_here = face_cascade.detectMultiScale(g, scaleFactor=1.1, minNeighbors=5)
    pass   # добавьте в counts количество найденных лиц

In [ ]:
#@title 🧪 Проверка задания 13 { display-mode: "form" }
def _check(tests):
    for i, (name, cond, hint) in enumerate(tests, 1):
        try:
            passed = bool(cond())
        except Exception:
            passed = False
        if not passed:
            print(f"❌ Тест {i} из {len(tests)}: {name}")
            print(f"   💡 {hint}")
            return False
        print(f"✅ Тест {i}: {name}")
    print(f"\n🎉 Задание 13 выполнено! Пройдено тестов: {len(tests)} из {len(tests)}")
    return True

if _check([
    ('число для каждого кадра', lambda: len(counts) == len(video2), 'counts.append(...) внутри цикла'),
    ('на каждом кадре 2 лица', lambda: all(c == 2 for c in counts), 'Количество — len(found_here)'),
]):
    print("Лиц по кадрам:", counts)

<details><summary>💡 Подсказка</summary>

Количество элементов списка — `len(...)`.
</details>
<details><summary>🆘 Почти готовый ответ</summary>

```python
counts.append(len(___))
```
</details>

---
# Часть 5. КТ — на своём видео (2 балла LXP)

| Задание LXP | Что сделать | Балл |
|---|---|---|
| «Записать в папку video только вырезанные лица» | Своё видео → файл `video/my_faces.mp4`, в котором только ваше лицо 200×200 | 1 |
| «Распознавание сразу нескольких лиц» | Видео, где в кадре 2 и больше лиц (сосед, фото на телефоне), рамки у всех + надпись `Faces: N` → `video/my_multi.mp4` | 1 |

**Сдать:** ссылку на ноутбук и два файла из папки `video`.

1. Запишите видео ячейкой ниже: `SOURCE = "webcam"` (5 секунд с камеры) или `"upload"` (свой файл до 10 секунд). Нет камеры — оставьте `"demo"`.
2. Ниже два пустых шаблона. Код уже был в заданиях 9–12 — соберите его.

In [ ]:
SOURCE = "demo"  # @param ["demo", "webcam", "upload"]

if SOURCE == "webcam":
    my_video = read_video(record_webcam(5))
elif SOURCE == "upload":
    from google.colab import files
    uploaded = files.upload()
    my_video = read_video(list(uploaded)[0])
else:
    my_video = make_face_video(faces=2)
show_video(my_video)

In [ ]:
# КТ 1: только лица → video/my_faces.mp4
# Подсказка: задание 11 (собрать лица 200×200) + задание 12 (записать файл)

In [ ]:
# КТ 2: все лица с рамками и надписью Faces: N → video/my_multi.mp4
# Подсказка: задание 10 (рамки на каждом кадре) + putText из задания 9.
# Размер для VideoWriter возьмите у кадра: h, w = my_video[0].shape[:2] → (w, h)

> 💻 **На компьютере колледжа** можно сделать то же самое вживую с веб-камерой: скрипт `faces_local.py` из папки урока на GitHub. Клавиши: `r` — запись лиц в `video/`, `q` — выход.

**Если лицо не находится:** смотрите прямо в камеру, включите свет, снимите очки или капюшон. Каскад Хаара плохо видит лица в профиль и с наклоном.

---
# Шпаргалка
| Что | Код |
|---|---|
| загрузить детектор | `face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")` |
| серая картинка | `gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)` |
| найти лица | `faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5)` |
| сколько лиц | `len(faces)` |
| рамка | `cv2.rectangle(img, (x, y), (x + w, y + h), (0, 255, 0), 3)` |
| вырезать | `face = img[y:y + h, x:x + w]` — **сначала y** |
| один размер | `cv2.resize(face, (200, 200))` |
| надпись | `cv2.putText(img, "Faces: 2", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)` |
| папка | `os.makedirs("video", exist_ok=True)` |
| записать видео | `out = cv2.VideoWriter("video/x.mp4", cv2.VideoWriter_fourcc(*"mp4v"), 10, (w, h))` → `out.write(f)` → `out.release()` |
| показать видео в Colab | `show_video(frames)` |